# 📈 Notebook 06 – Model Evaluation
**Goal:** judge the two baseline models fairly on the **unseen test set** using more than accuracy.

**Metrics explained**
- **Accuracy** – share of correct predictions (can be misleading with imbalance).
- **Precision** – when the model says *negative*, how often is it right?
- **Recall** – of all truly *negative* tweets, how many did the model find?
- **F1-score** – balance of precision and recall.
- **Macro F1** – average F1 of the 3 classes (each class counts equally → fair for imbalanced data).

## 1. Import Libraries and Load Models

In [ ]:
# Import evaluation tools
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 130)

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

In [ ]:
# Load the cleaned dataset and recreate the SAME train/test split that was made in Notebook 4
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv(CLEANED_PATH)
except FileNotFoundError:
    raise FileNotFoundError("cleaned_tweets.csv not found. Please run Notebook 03 first.")

df = df.dropna(subset=["cleaned_text", "airline_sentiment"]).reset_index(drop=True)

X = df["cleaned_text"]          # input feature
y = df["airline_sentiment"]     # target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))

In [ ]:
# Load the vectorizer and the two baseline models saved in Notebooks 04 and 05
try:
    tfidf    = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")
    nb_model = joblib.load(MODELS_DIR / "baseline_naive_bayes.pkl")
    lr_model = joblib.load(MODELS_DIR / "baseline_logistic_regression.pkl")
except FileNotFoundError as e:
    raise FileNotFoundError(f"{e}\nRun Notebooks 04 and 05 first.")

X_test_tfidf = tfidf.transform(X_test)
models = {"Naive Bayes": nb_model, "Logistic Regression": lr_model}
CLASSES = list(lr_model.classes_)             # ['negative', 'neutral', 'positive']
predictions = {name: m.predict(X_test_tfidf) for name, m in models.items()}
print("Classes:", CLASSES, "| Test samples:", len(y_test))

## 2. Overall Metrics Comparison

In [ ]:
# Accuracy, precision, recall and F1 for both models (macro = unweighted average over the 3 classes)
rows = []
for name, pred in predictions.items():
    rows.append({
        "Model": name,
        "Accuracy":        accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro"),
        "Recall (macro)":    recall_score(y_test, pred, average="macro"),
        "F1 (macro)":        f1_score(y_test, pred, average="macro"),
        "F1 (weighted)":     f1_score(y_test, pred, average="weighted"),
    })
metrics_df = pd.DataFrame(rows).set_index("Model").round(4)
metrics_df

In [ ]:
# Visual comparison of the main metrics
metrics_df[["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]].T.plot(
    kind="bar", figsize=(9, 5), color=["#3498db", "#e67e22"], rot=0)
plt.title("Baseline Model Comparison (Test Set)")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.legend(title="Model")
plt.show()

## 3. Classification Reports (per class)

In [ ]:
# Detailed precision / recall / F1 for each sentiment class
for name, pred in predictions.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(y_test, pred, digits=3))

In [ ]:
# Per-class F1-score chart (shows which sentiment is hardest to predict)
f1_by_class = pd.DataFrame({
    name: f1_score(y_test, pred, average=None, labels=CLASSES) for name, pred in predictions.items()
}, index=CLASSES)

f1_by_class.plot(kind="bar", figsize=(8, 4), color=["#3498db", "#e67e22"], rot=0)
plt.title("F1-score per Sentiment Class")
plt.ylim(0, 1)
plt.ylabel("F1")
plt.show()
f1_by_class.round(3)

## 4. Confusion Matrices

In [ ]:
# Confusion matrix: rows = true label, columns = predicted label (counts)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=CLASSES)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{name} – Counts")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrices_counts.png", dpi=150)
plt.show()

In [ ]:
# Normalised confusion matrix: each ROW sums to 100% = recall per class
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=CLASSES, normalize="true") * 100
    sns.heatmap(cm, annot=True, fmt=".1f", cmap="Greens", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{name} – % of each true class")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrices_percent.png", dpi=150)
plt.show()

**How to read it:** the diagonal is correct predictions. Look at the off-diagonal cells – *neutral* tweets are usually the most confused class because neutral language is closest to both sides.

## 5. ROC Curves and AUC (one-vs-rest)

In [ ]:
# ROC curve for each class: how well can the model rank tweets of that class above the others?
y_test_bin = label_binarize(y_test, classes=CLASSES)
colors = {"negative": "#e74c3c", "neutral": "#f5b041", "positive": "#2ecc71"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
auc_rows = []
for ax, (name, model) in zip(axes, models.items()):
    proba = model.predict_proba(X_test_tfidf)
    for i, cls in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        score = auc(fpr, tpr)
        auc_rows.append({"Model": name, "Class": cls, "AUC": score})
        ax.plot(fpr, tpr, color=colors[cls], label=f"{cls} (AUC = {score:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.set_title(f"{name} – ROC")
    ax.set_xlabel("False Positive Rate")
    ax.legend(loc="lower right")
axes[0].set_ylabel("True Positive Rate")
plt.tight_layout()
plt.show()
pd.DataFrame(auc_rows).pivot(index="Class", columns="Model", values="AUC").round(3)

## 6. What Words Drive the Predictions?

In [ ]:
# Logistic Regression: the words with the biggest positive weights for each class
feature_names = tfidf.get_feature_names_out()
coef = lr_model.coef_                      # shape: (3 classes, n_features)

top_lr = {}
for i, cls in enumerate(lr_model.classes_):
    top_idx = np.argsort(coef[i])[-12:][::-1]
    top_lr[cls] = [feature_names[j] for j in top_idx]
pd.DataFrame(top_lr, index=range(1, 13))

In [ ]:
# Naive Bayes: words that are much more likely in one class than in the others
log_prob = nb_model.feature_log_prob_
top_nb = {}
for i, cls in enumerate(nb_model.classes_):
    others = np.delete(log_prob, i, axis=0).mean(axis=0)
    score = log_prob[i] - others
    top_nb[cls] = [feature_names[j] for j in np.argsort(score)[-12:][::-1]]
pd.DataFrame(top_nb, index=range(1, 13))

In [ ]:
# Plot the strongest words for the NEGATIVE and POSITIVE class from Logistic Regression
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, cls, color in zip(axes, ["negative", "positive"], ["#e74c3c", "#2ecc71"]):
    i = list(lr_model.classes_).index(cls)
    idx = np.argsort(coef[i])[-15:]
    ax.barh([feature_names[j] for j in idx], coef[i][idx], color=color)
    ax.set_title(f"Top words for '{cls}'")
    ax.set_xlabel("Logistic Regression weight")
plt.tight_layout()
plt.show()

## 7. Error Analysis – Where Does the Best Model Fail?

In [ ]:
# Collect the Logistic Regression mistakes together with the ORIGINAL tweet text and model confidence
proba_lr = lr_model.predict_proba(X_test_tfidf)
errors = pd.DataFrame({
    "original_text": df.loc[X_test.index, "text"].values,
    "cleaned_text":  X_test.values,
    "actual":        y_test.values,
    "predicted":     predictions["Logistic Regression"],
    "confidence":    proba_lr.max(axis=1).round(3),
})
errors = errors[errors["actual"] != errors["predicted"]]
print(f"Misclassified tweets: {len(errors):,} of {len(y_test):,} ({len(errors)/len(y_test):.1%})")

In [ ]:
# Which kinds of mistakes happen most often?
pd.crosstab(errors["actual"], errors["predicted"], margins=True)

In [ ]:
# The most CONFIDENT mistakes are the most interesting: label noise, sarcasm or missing context
errors.sort_values("confidence", ascending=False)[["original_text", "actual", "predicted", "confidence"]].head(10)

**Typical reasons for errors:** sarcasm (*"great, another delay"*), mixed feelings, short questions with no sentiment words, and some noisy human labels. A bag-of-words model cannot fully understand context, which is a known limitation to mention in interviews.

## 8. Choose the Better Baseline and Save the Report

In [ ]:
# Pick the model with the higher macro F1 (the fairest metric for imbalanced classes) and save the table
best_name = metrics_df["F1 (macro)"].idxmax()
print(f"🏆 Best baseline model: {best_name}")
print(metrics_df.loc[best_name].to_string())

metrics_df.to_csv(REPORTS_DIR / "baseline_model_comparison.csv")
errors.head(200).to_csv(REPORTS_DIR / "baseline_misclassified_examples.csv", index=False)
print("\n✅ Reports saved in", REPORTS_DIR)

## ✅ Summary
- Compared Naive Bayes and Logistic Regression using accuracy, precision, recall, macro F1, confusion matrices and ROC/AUC.
- Neutral is normally the hardest class; negative the easiest (it has the most training data).
- Error analysis shows where bag-of-words models struggle (sarcasm, context).

**Next:** Notebook 07 – Model Optimization (hyperparameter tuning).